In [ ]:

#@title ⚙️ نصب و دانلود (کتابخانه‌ها + مدل OmniVoice)

HF_TOKEN = "" #@param {type:"string"}

import os

if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
    print('✅ توکن Hugging Face تنظیم شد.')
else:
    os.environ.pop('HF_TOKEN', None)
    print('ℹ️ بدون توکن Hugging Face ادامه می‌دهیم.')

import base64
encoded_text = "Tm90ZWJvb2sgTWFkZSBCeSBhaWdvbGRlbg=="
decoded_text = base64.b64decode(encoded_text.encode()).decode()
print(decoded_text)
print('='*30)

!pip install -q -U python-docx pydub ipywidgets
!pip install -q omnivoice
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg

import torch
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'Device: {device}')

if torch.cuda.is_available():
    print(
        f'GPU: {torch.cuda.get_device_name(0)} '
        f'({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)'
    )

print('در حال دانلود و لود مدل OmniVoice (اولین بار حدود ۳ تا ۵ گیگابایت دانلود می‌کند)...')

omni_model = OmniVoice.from_pretrained(
    'k2-fsa/OmniVoice',
    device_map=device,
    dtype=dtype
)

print('✅ همه چیز آماده‌ست، مدل لود شد!')

In [ ]:

#@title 📂 انتخاب فایل متنی (گوگل درایو یا حافظه داخلی)

import os
import time

#@markdown اگر این گزینه را فعال کنید، به گوگل درایو وصل نمی‌شویم و فایل را از حافظه داخلی آپلود می‌کنید.
Use_Local_File = True #@param {type:"boolean"}

SELECTED_DOC_PATH = None

TEXT_EXTENSIONS = (
    '.txt',
    '.docx',
    '.md',
    '.rtf',
    '.csv',
    '.srt',
    '.vtt',
    '.sub',
    '.ass',
    '.json',
    '.xml'
)

if Use_Local_File:

    from google.colab import files
    import ipywidgets as widgets
    from IPython.display import display

    print('📁 لطفاً فایل متنی خود را انتخاب کنید...')

    uploaded = files.upload()

    if not uploaded:
        raise RuntimeError('❌ فایلی انتخاب نشد!')

    fname = list(uploaded.keys())[0]

    if not fname.lower().endswith(TEXT_EXTENSIONS):
        raise RuntimeError(
            '❌ این فایل یک فرمت متنی پشتیبانی‌شده نیست!\n'
            f'فرمت‌های مجاز: {", ".join(TEXT_EXTENSIONS)}'
        )

    SELECTED_DOC_PATH = os.path.abspath(fname)

    print(f'✅ فایل انتخاب شد: {SELECTED_DOC_PATH}')

else:

    from google.colab import drive
    import ipywidgets as widgets
    from IPython.display import display

    print('🔄 در حال اتصال به Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    ROOT_DIR = '/content/drive/MyDrive'
    current_dir = [ROOT_DIR]

    path_label = widgets.HTML()
    listing = widgets.Select(
        rows=14,
        layout=widgets.Layout(width='100%')
    )

    btn_open = widgets.Button(
        description='📂 باز کردن / تأیید انتخاب',
        button_style='primary'
    )

    btn_up = widgets.Button(
        description='⬆️ پوشه بالاتر'
    )

    status = widgets.HTML()

    def refresh_listing():

        path_label.value = (
            f'<b>📍 مسیر فعلی:</b> {current_dir[0]}'
        )

        try:
            entries = sorted(
                os.listdir(current_dir[0]),
                key=lambda x: x.lower()
            )

        except Exception as e:

            status.value = (
                f'<span style="color:red">'
                f'❌ خطا در خواندن پوشه: {e}'
                f'</span>'
            )

            listing.options = []
            return

        options = []

        for e in entries:

            full = os.path.join(
                current_dir[0],
                e
            )

            if os.path.isdir(full):

                options.append(
                    f'📁 {e}'
                )

            elif e.lower().endswith(TEXT_EXTENSIONS):

                options.append(
                    f'📄 {e}'
                )

        listing.options = options

        if not options:

            status.value = (
                '<span style="color:#888">'
                '📭 در این پوشه فایل متنی پشتیبانی‌شده‌ای وجود ندارد.'
                '</span>'
            )

        else:

            status.value = ''

    def on_up_clicked(b):

        if os.path.normpath(current_dir[0]) == os.path.normpath(ROOT_DIR):
            status.value = (
                '<span style="color:orange">'
                '⚠️ شما در پوشه اصلی Google Drive هستید.'
                '</span>'
            )
            return

        parent = os.path.dirname(
            current_dir[0].rstrip('/')
        )

        if parent and len(parent) >= len(ROOT_DIR):

            current_dir[0] = parent
            refresh_listing()

    def on_open_clicked(b):

        global SELECTED_DOC_PATH

        if not listing.value:

            status.value = (
                '<span style="color:orange">'
                '⚠️ لطفاً ابتدا یک فایل را انتخاب کنید.'
                '</span>'
            )

            return

        name = listing.value[2:].strip()

        full = os.path.join(
            current_dir[0],
            name
        )

        if os.path.isdir(full):

            current_dir[0] = full
            refresh_listing()
            return

        if full.lower().endswith(TEXT_EXTENSIONS):

            SELECTED_DOC_PATH = full

            status.value = (
                '<div style="margin-top:8px; padding:10px; '
                'border-radius:6px; background:#e8f5e9; '
                'color:#1b5e20; font-weight:bold;">'
                '⏳ در حال انتخاب فایل...'
                '</div>'
            )

            time.sleep(0.5)

            status.value = (
                '<div style="margin-top:8px; padding:10px; '
                'border-radius:6px; background:#e8f5e9; '
                'color:#1b5e20; font-weight:bold;">'
                '✅ فایل با موفقیت انتخاب و تأیید شد.<br>'
                f'📄 {name}'
                '</div>'
            )

            print(
                f'✅ فایل انتخاب شد:\n{SELECTED_DOC_PATH}'
            )

    btn_up.on_click(on_up_clicked)
    btn_open.on_click(on_open_clicked)

    refresh_listing()

    display(
        path_label,
        listing,
        widgets.HBox([
            btn_up,
            btn_open
        ]),
        status
    )

    print(
        'راهنما: پوشه‌ها را انتخاب کنید و روی '
        '«📂 باز کردن / تأیید انتخاب» بزنید تا وارد آن‌ها شوید. '
        'برای فایل متنی نیز همین دکمه، انتخاب نهایی فایل را انجام می‌دهد.'
    )

In [ ]:
#@title 🎙️ آپلود صدای مرجع برای کلون
from google.colab import files
import os

#@markdown یک فایل صوتی کوتاه (۵ تا ۱۰ ثانیه، گفتار واضح و بدون نویز زمینه) از صدای مورد نظر آپلود کنید.
#@markdown اگر متن گفته‌شده در فایل را اینجا بنویسید، پردازش دقیق‌تر و سریع‌تر انجام می‌شود؛
#@markdown در غیر این‌صورت خالی بگذارید تا خودش متن را به‌صورت خودکار تشخیص دهد.
Ref_Text = '' #@param {type:"string"}
#@markdown ---
#@markdown #### 🗣️ تنظیمات تولید صدا (OmniVoice)
OmniVoice_Steps = 32 #@param {type:"slider", min:8, max:64, step:8}
OmniVoice_Speed = 1.0 #@param {type:"slider", min:0.5, max:2.0, step:0.1}

for f in ['ref_voice.wav', 'ref_voice.mp3']:
    if os.path.exists(f):
        os.remove(f)

print('📁 لطفاً فایل صوتی صدای مرجع را آپلود کنید:')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

fname = list(uploaded.keys())[0]
ext = fname.rsplit('.', 1)[-1].lower()
REF_AUDIO_PATH = f'ref_voice.{ext}'
os.rename(fname, REF_AUDIO_PATH)
REF_TEXT = Ref_Text

print(f'✅ صدای مرجع ذخیره شد: {REF_AUDIO_PATH}')
print(f'📝 متن مرجع: {"[خودکار با Whisper]" if not REF_TEXT.strip() else REF_TEXT}')
print(f'🗣️ گام‌های دیفیوژن: {OmniVoice_Steps} | ⚡ سرعت گفتار: {OmniVoice_Speed}x')

In [ ]:
#@title ▶️ شروع فرآیند: خواندن فایل با صدای کلون‌شده
import os
import torch, torchaudio
from pydub import AudioSegment
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML

YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"  # در صورت نیاز آدرس واقعی کانال را جایگزین کنید

def extract_paragraphs(path):
    ext = path.rsplit('.', 1)[-1].lower()
    if ext == 'docx':
        import docx
        doc = docx.Document(path)
        return [p.text.strip() for p in doc.paragraphs if p.text.strip()]
    else:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            raw = f.read()
        return [line.strip() for line in raw.splitlines() if line.strip()]

def generate_track(paragraphs, output_file):
    os.makedirs('voice_clone_segments', exist_ok=True)
    final_audio = AudioSegment.empty()
    SILENCE_BETWEEN_MS = 350
    for i, text in enumerate(paragraphs):
        seg_path = f'voice_clone_segments/seg_{i+1}.wav'
        ok = False
        for attempt in range(1, 4):
            try:
                kwargs = dict(
                    text=text,
                    ref_audio=REF_AUDIO_PATH,
                    num_step=int(OmniVoice_Steps),
                    speed=float(OmniVoice_Speed),
                )
                if REF_TEXT.strip():
                    kwargs['ref_text'] = REF_TEXT
                audio = omni_model.generate(**kwargs)
                audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
                if audio_tensor.dim() == 1:
                    audio_tensor = audio_tensor.unsqueeze(0)
                torchaudio.save(seg_path, audio_tensor.cpu(), 24000)
                print(f'  ✅ [{i+1}/{len(paragraphs)}] خوانده شد: "{text[:40]}..."')
                ok = True
                break
            except Exception as e:
                if attempt == 3:
                    print(f'  ⚠️ [{i+1}] خطا بعد از ۳ تلاش، این پاراگراف رد شد: {e}')
        if ok and os.path.exists(seg_path):
            final_audio += AudioSegment.from_file(seg_path)
            final_audio += AudioSegment.silent(duration=SILENCE_BETWEEN_MS)
    final_audio.export(output_file, format='mp3', bitrate='192k')
    return output_file

def show_result_widgets(output_file):
    print(f'🎉 تراک نهایی آماده شد: {output_file}')

    btn_download = widgets.Button(description='⬇️ دانلود تراک نهایی', button_style='success')
    def on_download_clicked(b):
        files.download(output_file)
    btn_download.on_click(on_download_clicked)
    display(btn_download)

    display(HTML(
        f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
        f'style="display:inline-block;margin-top:10px;padding:10px 22px;background:#FF0000;'
        f'color:#fff;border-radius:6px;text-decoration:none;font-family:sans-serif;font-weight:bold;">'
        f'🔔  ما را در یوتیوب دنبال کنید</a>'
    ))

if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH:
    raise RuntimeError('❌ ابتدا در سلول «انتخاب فایل متنی» یک فایل انتخاب کنید!')
if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

print(f'📄 در حال خواندن متن از: {SELECTED_DOC_PATH}')
paragraphs = extract_paragraphs(SELECTED_DOC_PATH)
if not paragraphs:
    raise RuntimeError('❌ متنی در فایل یافت نشد!')
print(f'✅ {len(paragraphs)} پاراگراف/خط استخراج شد.')

output_file = generate_track(paragraphs, 'cloned_voice_docx_reading.mp3')
show_result_widgets(output_file)

In [ ]:

#@title ✏️ اصلاح دستی تلفظ در متن (جایگزینی مستقیم عبارت‌ها)
import os
import ipywidgets as widgets
from IPython.display import display

if 'SELECTED_DOC_PATH' not in globals() or not SELECTED_DOC_PATH or not os.path.exists(SELECTED_DOC_PATH):
    raise RuntimeError('❌ ابتدا سلول «انتخاب فایل متنی» را اجرا کنید تا یک فایل انتخاب شود!')

CURRENT_FILE = SELECTED_DOC_PATH
FILE_EXT = CURRENT_FILE.rsplit('.', 1)[-1].lower()

def _read_lines():
    if FILE_EXT == 'docx':
        import docx
        doc = docx.Document(CURRENT_FILE)
        return doc, [p.text for p in doc.paragraphs]
    else:
        with open(CURRENT_FILE, 'r', encoding='utf-8', errors='ignore') as f:
            return None, f.read().splitlines()

def _write_lines(doc, lines):
    if FILE_EXT == 'docx':
        for p, new_text in zip(doc.paragraphs, lines):
            for run in list(p.runs):
                run.text = ''
            if p.runs:
                p.runs[0].text = new_text
            else:
                p.add_run(new_text)
        doc.save(CURRENT_FILE)
    else:
        with open(CURRENT_FILE, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

wrong_word = widgets.Text(description='کلمه اشتباه:', placeholder='مثلاً: نکته', layout=widgets.Layout(width='420px'))
correct_word = widgets.Text(description='عبارت درست:', placeholder='مثلاً: نُکته', layout=widgets.Layout(width='420px'))
btn_replace = widgets.Button(description='🔁 Replace', button_style='warning')
log_output = widgets.Output()
path_info = widgets.HTML(f'<b>📄 فایل فعلی:</b> {CURRENT_FILE}')

def on_replace_clicked(b):
    with log_output:
        log_output.clear_output()
        w = wrong_word.value.strip()
        c = correct_word.value.strip()
        if not w or not c:
            print('⚠️ هر دو فیلد را پر کنید.')
            return
        doc, lines = _read_lines()
        count = sum(line.count(w) for line in lines)
        if count == 0:
            print(f'ℹ️ عبارت «{w}» در فایل پیدا نشد.')
            return
        new_lines = [line.replace(w, c) for line in lines]
        _write_lines(doc, new_lines)
        print(f'✅ {count} مورد از «{w}» با «{c}» جایگزین شد و در فایل ذخیره شد.')
        wrong_word.value = ''
        correct_word.value = ''

btn_replace.on_click(on_replace_clicked)
display(path_info, wrong_word, correct_word, btn_replace, log_output)
print('راهنما: بعد از گوش دادن به فایل صوتی، کلمه‌ی اشتباه‌خوانده‌شده را در فیلد اول و شکل درست (با اعراب) را در فیلد دوم بنویس و Replace بزن. برای هر اشتباه دیگه همین کار رو تکرار کن. در پایان دوباره سلول «شروع فرآیند» رو اجرا کن تا صدای اصلاح‌شده تولید بشه.')

In [ ]:

#@title 🧪 (اختیاری) تست دستی بدون فایل — وارد کردن متن مستقیم

import os
import torch
import torchaudio
from pydub import AudioSegment
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, Audio

Manual_Text = "" #@param {type:"string"}

YOUTUBE_CHANNEL_URL = "https://www.youtube.com/@aigolden?sub_confirmation=1"

if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
    raise RuntimeError('❌ ابتدا سلول «آپلود صدای مرجع» را اجرا کنید!')

paragraphs = [line.strip() for line in Manual_Text.splitlines() if line.strip()]

if not paragraphs:
    raise RuntimeError('❌ متنی وارد نشده است!')

os.makedirs('voice_clone_segments_manual', exist_ok=True)

final_audio = AudioSegment.empty()
SILENCE_BETWEEN_MS = 350

for i, text in enumerate(paragraphs):
    seg_path = f'voice_clone_segments_manual/seg_{i+1}.wav'
    ok = False

    for attempt in range(1, 4):
        try:
            kwargs = dict(
                text=text,
                ref_audio=REF_AUDIO_PATH,
                num_step=int(OmniVoice_Steps),
                speed=float(OmniVoice_Speed),
            )

            if REF_TEXT.strip():
                kwargs['ref_text'] = REF_TEXT

            audio = omni_model.generate(**kwargs)

            audio_tensor = (
                audio[0]
                if isinstance(audio[0], torch.Tensor)
                else torch.tensor(audio[0])
            )

            if audio_tensor.dim() == 1:
                audio_tensor = audio_tensor.unsqueeze(0)

            torchaudio.save(
                seg_path,
                audio_tensor.cpu(),
                24000
            )

            print(
                f'  ✅ [{i+1}/{len(paragraphs)}] '
                f'خوانده شد: "{text[:40]}..."'
            )

            ok = True
            break

        except Exception as e:
            if attempt == 3:
                print(
                    f'  ⚠️ [{i+1}] خطا بعد از ۳ تلاش، '
                    f'این بخش رد شد: {e}'
                )

    if ok and os.path.exists(seg_path):
        final_audio += AudioSegment.from_file(seg_path)
        final_audio += AudioSegment.silent(
            duration=SILENCE_BETWEEN_MS
        )

output_file = 'cloned_voice_manual_test.mp3'

final_audio.export(
    output_file,
    format='mp3',
    bitrate='192k'
)

print(f'🎉 تراک آزمایشی آماده شد: {output_file}')
print('▶️ پخش خودکار تراک...')

# پخش خودکار بعد از اتمام کامل تولید فایل
display(Audio(
    output_file,
    autoplay=True
))

btn_download = widgets.Button(
    description='⬇️ دانلود تراک آزمایشی',
    button_style='success'
)

def on_download_clicked(b):
    files.download(output_file)

btn_download.on_click(on_download_clicked)

display(btn_download)

display(HTML(
    f'<a href="{YOUTUBE_CHANNEL_URL}" target="_blank" '
    f'style="display:inline-block;margin-top:10px;padding:10px 22px;'
    f'background:#FF0000;color:#fff;border-radius:6px;'
    f'text-decoration:none;font-family:sans-serif;font-weight:bold;">'
    f'🔔 ما را در یوتیوب دنبال کنید</a>'
))

In [ ]:
#@title پاکسازی
!find /content -mindepth 1 -maxdepth 1 ! -name 'drive' -exec rm -rf {} +


>⚠️ توجه

> این تگها بعضیاشون ضعیف عمل میکنن
| تگ | واکنش صوتی |
|---|---|
| `[laughter]` | خنده |
| `[sigh]` | آه کشیدن / نفس عمیق از خستگی |
| `[confirmation-en]` | صدای تاییدی (هوم / باشه) |
| `[question-en]` |«en» صدای سؤالی با مصوت  |
| `[question-ah]` |«ah» صدای سؤالی با مصوت  |
| `[question-oh]` |«oh» صدای سؤالی با مصوت  |
| `[question-ei]` |«ei» صدای سؤالی با مصوت  |
| `[question-yi]` |«yi» صدای سؤالی با مصوت  |
| `[surprise-ah]` |«ah» صدای تعجبی با مصوت  |
| `[surprise-oh]` |«oh» صدای تعجبی با مصوت  |
| `[surprise-wa]` |«wa» صدای تعجبی با مصوت  |
| `[surprise-yo]` |«yo» صدای تعجبی با مصوت  |
| `[dissatisfaction-hnn]` | صدای نارضایتی («هنن») |